# 📖 phonetok — Usage Examples

**orthography2ipa** provides linguistically motivated grapheme→IPA mappings for 109 language codes.
**phonetok** is a language-agnostic tokenizer that segments text into graphemes and expands them into IPA via beam search.

```bash
pip install -e .
```

## 1. Setup & Quick Start

In [28]:
import orthography2ipa
from orthography2ipa import get
from orthography2ipa.phonetok import PhonetokTokenizer, TokenKind

print(f"orthography2ipa v{orthography2ipa.__version__}")
print(f"Available language codes: {len(orthography2ipa.available_codes())}")
print(f"First 20: {orthography2ipa.available_codes()[:20]}")

orthography2ipa v0.1.0
Available language codes: 109
First 20: ['an', 'an-x-occidental', 'an-x-oriental', 'ar', 'ast', 'ast-ES-x-leon', 'ast-PT-x-guadramil', 'ast-PT-x-rionor', 'ast-x-occidental', 'ast-x-oriental', 'ca', 'ca-x-balear', 'ca-x-nord', 'ca-x-occidental', 'ca-x-valencia', 'cs', 'da', 'de', 'el', 'en']


## 2. Loading a Language Spec

Each `LanguageSpec` contains the full grapheme→IPA table and allophone inventory.

In [29]:
pt = get("pt-BR")
print(f"Language: {pt.name} ({pt.code})")
print(f"Family:   {pt.family}")
print(f"Parent:   {pt.parent}")
print(f"Graphemes: {len(pt.graphemes)} entries")
print(f"Allophones: {len(pt.allophones)} entries")
print()

# Look up specific graphemes
print("pt-BR grapheme table samples:")
for g in ["ch", "lh", "nh", "rr", "ão", "õe", "t", "d"]:
    print(f"  ⟨{g}⟩ → {pt.graphemes[g]}")

Language: Brazilian Portuguese (pt-BR)
Family:   Romance
Parent:   pt
Graphemes: 57 entries
Allophones: 37 entries

pt-BR grapheme table samples:
  ⟨ch⟩ → ['ʃ']
  ⟨lh⟩ → ['ʎ']
  ⟨nh⟩ → ['ɲ']
  ⟨rr⟩ → ['ʁ']
  ⟨ão⟩ → ['ɐ̃w̃']
  ⟨õe⟩ → ['õj̃']
  ⟨t⟩ → ['t', 'tʃ']
  ⟨d⟩ → ['d', 'dʒ']


## 3. Basic Tokenization

`PhonetokTokenizer` segments text into tokens using **maximal munch** (longest grapheme match first).  
It handles digraphs, trigraphs, whitespace, punctuation, and digits automatically.

In [30]:
tok = PhonetokTokenizer(get("pt-BR"))
print(tok)
print()

tokens = tok.tokenize("Chuva grande, né? 100%")
print(f"{'KIND':<14} {'GRAPHEME':<10} {'IPA':30} {'POS':>4}")
print("─" * 62)
for t in tokens:
    ipa_str = " | ".join(t.ipa) if t.ipa else "—"
    print(f"{t.kind.name:<14} {t.grapheme!r:<10} {ipa_str:30} {t.position:>4}")

PhonetokTokenizer(lang='pt-BR', graphemes=57, vocab_size=64)

KIND           GRAPHEME   IPA                             POS
──────────────────────────────────────────────────────────────
GRAPHEME       'ch'       ʃ                                 0
GRAPHEME       'u'        u                                 2
GRAPHEME       'v'        v                                 3
GRAPHEME       'a'        a | ɐ                             4
WHITESPACE     ' '        —                                 5
GRAPHEME       'g'        ɡ | ʒ                             6
GRAPHEME       'r'        ɾ | h | x                         7
GRAPHEME       'a'        a | ɐ                             8
GRAPHEME       'n'        n                                 9
GRAPHEME       'd'        d | dʒ                           10
GRAPHEME       'e'        e | ɛ | ə                        11
PUNCTUATION    ','        —                                12
WHITESPACE     ' '        —                                13
GRAPHEM

## 4. Digraph & Trigraph Priority

The trie-based matcher always picks the **longest** grapheme.  
Catalan `l·l` (ela geminada), Basque `tx`, Portuguese `lh`/`nh`/`ch` all work automatically.

In [31]:
# Catalan: ⟨l·l⟩ trigraph must beat ⟨l⟩ + ⟨·⟩ + ⟨l⟩
tok_ca = PhonetokTokenizer(get("ca"))
for t in tok_ca.grapheme_tokens("col·laboració"):
    print(f"  ⟨{t.grapheme}⟩ → {t.ipa}")

print()

# Basque: ⟨tx⟩ digraph must beat ⟨t⟩ + ⟨x⟩
tok_eu = PhonetokTokenizer(get("eu"))
for t in tok_eu.grapheme_tokens("etxean"):
    print(f"  ⟨{t.grapheme}⟩ → {t.ipa}")

print()

# Mirandese: ⟨lh⟩ word-initial palatalisation
tok_mwl = PhonetokTokenizer(get("mwl"))
for t in tok_mwl.grapheme_tokens("lhobu"):
    print(f"  ⟨{t.grapheme}⟩ → {t.ipa}")

  ⟨c⟩ → ('k', 's')
  ⟨o⟩ → ('o', 'ɔ', 'u')
  ⟨l·l⟩ → ('lː',)
  ⟨a⟩ → ('a', 'ə')
  ⟨b⟩ → ('b',)
  ⟨o⟩ → ('o', 'ɔ', 'u')
  ⟨r⟩ → ('ɾ', 'r')
  ⟨a⟩ → ('a', 'ə')
  ⟨c⟩ → ('k', 's')
  ⟨i⟩ → ('i',)
  ⟨ó⟩ → ('o',)

  ⟨e⟩ → ('e',)
  ⟨tx⟩ → ('tʃ',)
  ⟨e⟩ → ('e',)
  ⟨a⟩ → ('a',)
  ⟨n⟩ → ('n',)

  ⟨lh⟩ → ('ʎ',)
  ⟨o⟩ → ('o', 'ɔ')
  ⟨b⟩ → ('b',)
  ⟨u⟩ → ('u',)


## 5. IPA Beam Search

Many graphemes are **ambiguous** — English ⟨c⟩ can be /k/ or /s/, Portuguese ⟨r⟩ can be /ɾ/ or /h/.  
`ipa_beam()` enumerates all combinatorial paths, bounded by beam width.

Each path has a **score**: the canonical (first-listed) IPA gets cost 0, alternatives get +1 each.  
Lower score = more canonical.

In [32]:
tok_en = PhonetokTokenizer(get("en"))

# "the" has ⟨th⟩→[θ|ð] and ⟨e⟩→[ɛ|iː|ə|...]
paths = tok_en.ipa_beam("the", beam_width=10)
print("All IPA paths for 'the':")
for p in paths:
    print(f"  /{p.ipa}/  (score={p.score:.1f})")

print()
print(f"Best: /{tok_en.ipa_best('the')}/")

All IPA paths for 'the':
  /θɛ/  (score=0.0)
  /ðɛ/  (score=1.0)
  /θiː/  (score=1.0)
  /ðiː/  (score=2.0)
  /θə/  (score=2.0)
  /ðə/  (score=3.0)

Best: /θɛ/


## 6. Allophone Expansion

With `expand_allophones=True`, each phoneme further branches into its **surface realisations**.  
This models actual pronunciation variation — e.g. Carioca Portuguese coda /s/ → [ʃ] (chiado).

In [33]:
# Carioca (Rio): coda /s/ has palatal allophone [ʃ]
tok_rj = PhonetokTokenizer(get("pt-BR-x-rj"))

print("Without allophones:")
for p in tok_rj.ipa_beam("mas", beam_width=6):
    print(f"  /{p.ipa}/  score={p.score:.1f}")

print()
print("WITH allophones:")
for p in tok_rj.ipa_beam("mas", beam_width=12, expand_allophones=True):
    print(f"  [{p.ipa}]  score={p.score:.1f}")

Without allophones:
  /mas/  score=0.0
  /maz/  score=1.0
  /mɐs/  score=1.0
  /mɐz/  score=2.0

WITH allophones:
  [mas]  score=0.0
  [maʃ]  score=0.5
  [maz]  score=1.0
  [mɐs]  score=1.0
  [maʒ]  score=1.5
  [mɐʃ]  score=1.5
  [mɐz]  score=2.0
  [mɐʒ]  score=2.5


## 7. Comparing Dialects

Same word, different phonological systems. The tokenizer algorithm is identical — only the `LanguageSpec` changes.

In [34]:
word = "cerveza"
dialects = [
    ("es",                  "Castilian"),
    ("es-ES-x-andalusia-w", "W. Andalusian"),
    ("es-CU",               "Cuban"),
    ("es-MX",               "Mexican"),
    ("es-AR",               "Rioplatense"),
    ("es-CL",               "Chilean"),
]

print(f"{'Dialect':<20} {'Best IPA':<20} Alternatives")
print("─" * 60)
for code, label in dialects:
    t = PhonetokTokenizer(get(code))
    paths = t.ipa_beam(word, beam_width=4)
    best = paths[0].ipa
    alts = ", ".join(f"/{p.ipa}/" for p in paths[1:3])
    print(f"{label:<20} /{best}/<{'':>8} {alts}")

Dialect              Best IPA             Alternatives
────────────────────────────────────────────────────────────
Castilian            /keɾbeθa/<         /θeɾbeθa/
W. Andalusian        /keɾbesa/<         /keɾbeθa/, /seɾbesa/
Cuban                /keɾbesa/<         /seɾbesa/
Mexican              /keɾbesa/<         /seɾbesa/
Rioplatense          /keɾbesa/<         /seɾbesa/
Chilean              /keɾbesa/<         /seɾbesa/


## 8. Brazilian Portuguese Dialect Comparison

The key isoglosses: /t,d/ palatalisation, coda /s/ realisation, rhotic type.

In [35]:
word = "triste"
br_dialects = [
    "pt-BR-x-sp",       # Paulistano
    "pt-BR-x-rj",       # Carioca
    "pt-BR-x-caipira",  # Caipira (retroflex)
    "pt-BR-x-recife",   # Nordestino (no palatalisation)
    "pt-BR-x-sul",      # Gaúcho (trill)
]

print(f"{'Dialect':<35} Top-3 allophonic paths")
print("─" * 75)
for c in br_dialects:
    spec = get(c)
    t = PhonetokTokenizer(spec)
    paths = t.ipa_beam(word, beam_width=3, expand_allophones=True)
    alts = "  ".join(f"[{p.ipa}]" for p in paths[:3])
    print(f"{spec.name:<35} {alts}")

Dialect                             Top-3 allophonic paths
───────────────────────────────────────────────────────────────────────────
Paulistano Portuguese               [tɾiste]  [tɾistʃe]  [tʃɾiste]
Carioca Portuguese                  [tɾiste]  [tɾistʃe]  [tɾiʃte]
Caipira Portuguese                  [tɻiste]  [tɹiste]  [tɻistʃe]
Pernambucano Portuguese             [tɾiste]  [thiste]  [tɾistɛ]
Sulista/Gaúcho Portuguese           [tɾiste]  [tɾistʃe]  [tʃɾiste]


## 9. Multi-Script Support

The tokenizer is script-agnostic. It works on Latin, Cyrillic, Arabic, Devanagari, CJK, Kana — whatever the grapheme table provides.

In [36]:
examples = [
    ("ja", "さくら",    "Japanese"),
    ("ar", "كتاب",     "Arabic"),
    ("ru", "молоко",   "Russian"),
    ("hi", "नमस्ते",    "Hindi"),
    ("el", "ελληνικά", "Greek"),
]

for code, word, label in examples:
    t = PhonetokTokenizer(get(code))
    graphemes = [tk.grapheme for tk in t.grapheme_tokens(word)]
    ipa = t.ipa_best(word)
    print(f"{label:<12} {word:<10}  graphemes={graphemes}")
    print(f"{'':12} {'':10}  IPA: /{ipa}/")
    print()

Japanese     さくら         graphemes=['さ', 'く', 'ら']
                         IPA: /sakɯɾa/

Arabic       كتاب        graphemes=['ك', 'ت', 'ا', 'ب']
                         IPA: /ktʔb/

Russian      молоко      graphemes=['м', 'о', 'л', 'о', 'к', 'о']
                         IPA: /moloko/

Hindi        नमस्ते      graphemes=['न', 'म', 'स', '्', 'त', 'े']
                         IPA: /nmst̪eː/

Greek        ελληνικά    graphemes=['ε', 'λ', 'λ', 'η', 'ν', 'ι', 'κ', 'ά']
                         IPA: /ellinika/



## 10. Integer Encoding for ML Pipelines

`encode()` maps tokens to integer IDs. `decode()` reverses it.  
7 special tokens (`<pad>`, `<bos>`, `<eos>`, `<unk>`, `<ws>`, `<punct>`, `<digit>`) + all graphemes.

In [37]:
tok = PhonetokTokenizer(get("es"), add_bos=True, add_eos=True)

text = "¡Hola, mundo!"
ids = tok.encode(text)
decoded = tok.decode(ids)

print(f"Input:      {text!r}")
print(f"Token IDs:  {ids}")
print(f"Decoded:    {decoded!r}")
print(f"Vocab size: {tok.vocab_size}")
print()

# Show vocab excerpt
print("Vocabulary (first 17 entries):")
for key, idx in list(tok.vocab.items())[:17]:
    print(f"  {idx:3d}: {key!r}")

Input:      '¡Hola, mundo!'
Token IDs:  [1, 5, 22, 37, 33, 7, 5, 4, 35, 48, 36, 14, 37, 5, 2]
Decoded:    'hola mundo'
Vocab size: 68

Vocabulary (first 17 entries):
    0: '<pad>'
    1: '<bos>'
    2: '<eos>'
    3: '<unk>'
    4: '<ws>'
    5: '<punct>'
    6: '<digit>'
    7: 'a'
    8: 'ai'
    9: 'au'
   10: 'ay'
   11: 'b'
   12: 'c'
   13: 'ch'
   14: 'd'
   15: 'e'
   16: 'ei'


## 11. Full Sentences with Word Boundaries

Use `include_special=True` to preserve whitespace as IPA word separators.

In [38]:
tok = PhonetokTokenizer(get("pt-BR"))

sentences = [
    "Bom dia, como vai?",
    "Eu gosto de café.",
]

for sent in sentences:
    ipa = tok.ipa_best(sent, include_special=True, word_separator=" ")
    print(f"  {sent}")
    print(f"  → /{ipa}/")
    print()

  Bom dia, como vai?
  → /bom dia, komo vaj?/

  Eu gosto de café.
  → /ew ɡosto de kafɛ./



## 12. Browse All 109 Language Codes

In [39]:
families = orthography2ipa.available_families()

for fam in sorted(families):
    codes = families[fam]
    names = [f"{c} ({get(c).name})" for c in codes[:5]]
    more = f" … +{len(codes)-5} more" if len(codes) > 5 else ""
    print(f"{fam} ({len(codes)}): {', '.join(names)}{more}")

Asturleonese (8): ast (Asturian (Central)), ast-ES-x-leon (Leonese), ast-PT-x-guadramil (Guadramilês), ast-PT-x-rionor (Rionorês), ast-x-occidental (Western Asturian) … +3 more
Germanic (6): da (Danish), de (German), en (English), nl (Dutch), no (Norwegian) … +1 more
Hellenic (1): el (Modern Greek)
Indo-Aryan (1): hi (Hindi)
Iranian (1): fa (Persian (Farsi))
Isolate (6): eu (Basque (Euskara)), eu-x-bizkaiera (Biscayan Basque (Bizkaiera)), eu-x-gipuzkera (Gipuzkoan Basque (Gipuzkera)), eu-x-nafarra-beherea (Lower Navarrese Basque), eu-x-nafarra-garaia (Upper Navarrese Basque) … +1 more
Japonic (1): ja (Japanese)
Koreanic (1): ko (Korean)
Romance (76): an (Aragonese), an-x-occidental (Western Aragonese), an-x-oriental (Eastern Aragonese (Ribagorçan)), ca (Catalan), ca-x-balear (Balearic Catalan) … +71 more
Semitic (1): ar (Arabic (MSA))
Sinitic (1): zh (Mandarin Chinese)
Slavic (4): cs (Czech), pl (Polish), ru (Russian), uk (Ukrainian)
Turkic (1): tr (Turkish)
Uralic (1): fi (Finnish)


## 13. Advanced: Custom LanguageSpec

You can create a `LanguageSpec` from scratch for any language or constructed orthography.

In [40]:
from orthography2ipa.types import LanguageSpec

toy = LanguageSpec(
    code="x-toy",
    name="Toy Phonetic",
    family="Constructed",
    script="Latin",
    graphemes={
        "a": ["a"], "e": ["e"], "i": ["i"], "o": ["o"], "u": ["u"],
        "p": ["p"], "t": ["t"], "k": ["k"],
        "m": ["m"], "n": ["n"], "ng": ["ŋ"],
        "s": ["s"], "sh": ["ʃ"],
        "r": ["ɾ"], "l": ["l"],
    },
    allophones={
        "p": ["p", "pʰ"], "t": ["t", "tʰ"], "k": ["k", "kʰ"],
        "ɾ": ["ɾ", "r"],
    },
)

tok_toy = PhonetokTokenizer(toy)
print(tok_toy)
print()

for t in tok_toy.tokenize("shingle"):
    print(f"  {t}")

print()
print("IPA beam with allophones:")
for p in tok_toy.ipa_beam("shingle", beam_width=4, expand_allophones=True):
    print(f"  [{p.ipa}]  score={p.score:.1f}")

PhonetokTokenizer(lang='x-toy', graphemes=15, vocab_size=22)

  Token(GRAPHEME, 'sh', [ʃ], pos=0)
  Token(GRAPHEME, 'i', [i], pos=2)
  Token(GRAPHEME, 'ng', [ŋ], pos=3)
  Token(GRAPHEME, 'l', [l], pos=5)
  Token(GRAPHEME, 'e', [e], pos=6)

IPA beam with allophones:
  [ʃiŋle]  score=0.0
